# Neural Networks

## What Is a Neural Network?

A neural network is what you get when you chain multiple perceptrons together into layers. The output of one layer becomes the input to the next. Each layer learns to recognize increasingly abstract patterns: early layers might detect simple structures in the data, and later layers combine those to recognize more complex ones.

The layers you can't see from the outside — the ones between the raw inputs and the final prediction — are called **hidden layers**. A network with one hidden layer is sometimes called a "vanilla" or shallow network. Networks with many hidden layers are called **deep** networks — that's where the term "deep learning" comes from.

## Why Activation Functions Matter

Here's a critical point: if you stack layers of perceptrons but don't include any non-linearity, the whole stack collapses into a single linear operation. A linear function of a linear function is still linear — you've gained nothing from the extra layers.

**Activation functions** break this. After computing the weighted sum at each node, we pass the result through a non-linear function before sending it to the next layer. Common choices:

- **Binary step / Heaviside**: Outputs 0 or 1. Simple, but not differentiable at the threshold — which makes gradient-based training impossible.
- **Sigmoid / Logistic**: Squashes any input to a smooth curve between 0 and 1. Differentiable everywhere, good for output layers when you want a probability.
- **Tanh**: Like sigmoid but outputs between -1 and 1. Often works better in hidden layers because it's centered at zero.
- **ReLU (Rectified Linear Unit)**: Outputs 0 for negative inputs, and the input itself for positive values. Simple and fast; the most common choice in modern networks.
- **Leaky ReLU**: Like ReLU, but allows a small negative output for negative inputs. This prevents "dead neurons" that get stuck at zero.

The choice of activation function affects how well the network trains and what kinds of patterns it can learn.

## Forward Pass and Backpropagation

Training a neural network involves two alternating passes:

**Forward pass** — making a prediction: we push the input through the network layer by layer, computing each node's output using the current weights, until we reach the final prediction.

**Backward pass (backpropagation)** — learning from mistakes: we compare the prediction to the correct answer, compute how wrong we were (the loss), and then work *backwards* through the network to figure out how much each weight contributed to that error. We nudge each weight in the direction that reduces the error. The learning rate controls how large those nudges are.

This back-and-forth — predict, measure error, adjust weights — repeats until the network stops improving or we run out of patience.

## Epochs and Batch Size

**Epoch**: One complete pass through the entire training dataset. Networks typically need many epochs — sometimes hundreds or thousands — before they converge.

**Batch size**: Instead of updating weights after every single example (slow) or after seeing all examples at once (memory-intensive), we process examples in small **batches** and update after each batch. Smaller batches introduce more noise into the updates, which can actually help the network find better solutions by avoiding getting stuck in local minima. Larger batches are more stable but can overfit more easily.

**Stochastic gradient descent (SGD)** is the extreme case of batch size = 1: update after every single example. In practice, mini-batches of 32–256 examples are common.

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets

## The Dataset: Handwritten Digits

We'll use scikit-learn's built-in digits dataset. It contains 1,797 images of handwritten digits (0–9), each an 8x8 grid of pixel values. The task is to classify each image as the correct digit.

This is a good test for a neural network because the raw pixels carry no explicit information about which digit they represent — the network has to learn what combinations of pixel patterns correspond to each digit.

In [ ]:
digits = datasets.load_digits()
dir(digits)
print('\n')
print(type(digits.images))
print('\n')
print(type(digits.target))

## Inspecting the Data

`digits.images` holds the raw 8x8 pixel grids. `digits.data` holds the same data flattened into 64-element row vectors — that's the format we'll feed into the neural network, one pixel per input node. `digits.target` holds the true label (0 through 9) for each image.

In [ ]:
digits.images.shape
print('\n')
digits.data.shape
print('\n')
digits.target.shape

In [ ]:
digits.images[0]

In [ ]:
digits.data[0]

## Visualizing a Sample Image

Let's look at the first image to make the data concrete. We're plotting the 8x8 pixel grid in grayscale — you should be able to recognize a handwritten digit.

In [ ]:
plt.imshow(digits.images[0],cmap='binary')
plt.show()

In [ ]:
digits.target[0]

## A Sample of Each Digit

Before training, let's see what the data actually looks like — one example of each digit from 0 to 9. Each image is only 8×8 pixels, so they're quite coarse. The network will need to learn to recognize each digit from just 64 pixel values.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
shown = {}
for img, label in zip(digits.images, digits.target):
    if label not in shown:
        shown[label] = img
    if len(shown) == 10:
        break

for ax, digit in zip(axes.flat, range(10)):
    ax.imshow(shown[digit], cmap='binary')
    ax.set_title(f'Digit: {digit}', fontsize=10)
    ax.axis('off')

plt.suptitle('One example of each digit — 8×8 pixels each', fontsize=12)
plt.tight_layout()
plt.show()

## Network Architecture

Our neural network will have:
- **Input layer**: 64 nodes — one for each pixel in the 8x8 image
- **Hidden layer**: 20 nodes — these learn intermediate representations of digit features (edges, curves, corners, etc.)
- **Output layer**: 10 nodes — one for each possible digit (0 through 9). The output is a probability distribution over the 10 classes; the predicted digit is whichever class gets the highest probability.

More hidden nodes give the network more capacity to learn complex patterns, but also increase the risk of overfitting. 20 nodes is a reasonable middle ground for this dataset.

In [ ]:
X = digits.data
Y = digits.target

In [ ]:
X.shape
print('\n')
Y.shape

## Splitting Into Training and Test Sets

We hold back the last 897 images as a test set and train on the first 900. The test set lets us measure how well the network generalizes to examples it's never seen — this is the number that actually matters, not the training accuracy.

In [ ]:
X_train = X[:900]
Y_train = Y[:900]
X_test = X[900:]
Y_test = Y[900:]

## Configuring the Network

A few key parameters:

- `hidden_layer_sizes=(20,)`: One hidden layer with 20 nodes.
- `activation='logistic'`: Uses the sigmoid function at each hidden node. Outputs are between 0 and 1, which works well here.
- `solver='sgd'`: Trains using stochastic gradient descent — processes one example at a time rather than the whole dataset at once.
- `alpha=0`: No regularization penalty. We'll discuss what regularization does in a moment.
- `learning_rate_init=0.1`: Each weight update moves 10% of the way in the direction of the gradient. Too large and training overshoots; too small and it's painfully slow.
- `verbose=True`: Prints the loss at each epoch so we can watch it decrease during training.

In [ ]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(20,), activation='logistic', alpha=0,
                    solver='sgd', tol=1e-4, random_state=1,
                    learning_rate_init=.1, verbose=True)

## Training

With `verbose=True`, you'll see the loss printed after each epoch. Watch for it to decrease — that's the network getting better. If it plateaus or stops decreasing early, the network has converged. If it starts oscillating, the learning rate may be too high.

In [ ]:
mlp.fit(X_train, Y_train)

## Evaluating on the Test Set

Now we predict on the held-out test set. The accuracy score tells us what fraction of test images the network classified correctly. A network that just guessed randomly would get about 10% (1 in 10 digits). A well-trained network should do considerably better.

## What Overfitting Looks Like

Overfitting happens when a model learns the training data too well — including its noise and quirks — and then fails to generalize to new data. In practice, you'd see it as:

- Training loss continues to decrease epoch after epoch.
- Validation (or test) loss stops decreasing and starts climbing back up.

That divergence between training and validation loss is the signature of overfitting. The model has essentially memorized the training set rather than learning general patterns.

**How to fight overfitting:**

- **Early stopping**: Stop training when the validation loss stops improving, even if training loss is still going down.
- **Dropout**: During each training step, randomly "switch off" a fraction of the nodes (set their output to zero). This forces the network to learn redundant representations and prevents it from relying too heavily on any single path through the network. At prediction time, all nodes are active again. Dropout is the neural network equivalent of an ensemble method.
- **Weight decay (regularization)**: Add a penalty for having large weights. The `alpha` parameter in scikit-learn's MLPClassifier controls this. Larger alpha means stronger regularization, pushing weights toward zero and preventing the model from being too confident about any single feature.

In [ ]:
predictions = mlp.predict(X_test)
predictions

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(Y_test, predictions)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.plot(mlp.loss_curve_, color='steelblue', linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Loss Curve During Training')
plt.tight_layout()
plt.show()

## Where Did the Network Struggle? Confusion Matrix

A single accuracy number hides a lot. The confusion matrix shows the full picture: each row is a true digit, each column is a predicted digit. Diagonal cells are correct classifications; off-diagonal cells are mistakes. Bright off-diagonal squares reveal which pairs of digits the network confuses most.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(Y_test, predictions)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(range(10)); ax.set_yticklabels(range(10))
ax.set_xlabel('Predicted digit', fontsize=12)
ax.set_ylabel('True digit', fontsize=12)
ax.set_title('Confusion Matrix — Test Set\n(diagonal = correct classifications; off-diagonal = mistakes)',
             fontsize=12)
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=9,
                color='white' if cm[i, j] > cm.max() * 0.5 else 'black')
plt.tight_layout()
plt.show()

print('Most common misclassifications (true → predicted):')
off_diag = [(cm[i,j], i, j) for i in range(10) for j in range(10) if i != j and cm[i,j] > 0]
for count, true, pred in sorted(off_diag, reverse=True)[:5]:
    print(f'  {true} mistaken for {pred}: {count} time(s)')

## Interactive: Explore Training Parameters

Use the sliders below to try different network configurations and see how they affect test accuracy. This lets you build intuition for how architecture choices and training hyperparameters interact.

Things to try:
- Increase the number of hidden nodes — does accuracy improve? At what cost?
- Raise the learning rate — does training become unstable?
- Add a second hidden layer — does deeper help here?
- Increase alpha (regularization) — does it reduce overfitting?

Note: each run re-trains the network, so expect it to take a few seconds.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

out = widgets.Output()

hidden_nodes_slider = widgets.IntSlider(value=20, min=5, max=100, step=5,
                                         description='Hidden nodes', style={'description_width': 'initial'})
second_layer_slider = widgets.IntSlider(value=0, min=0, max=50, step=5,
                                         description='2nd layer nodes (0=none)', style={'description_width': 'initial'})
lr_slider = widgets.FloatLogSlider(value=0.1, base=10, min=-3, max=0, step=0.25,
                                    description='Learning rate', style={'description_width': 'initial'})
alpha_slider = widgets.FloatLogSlider(value=0.0001, base=10, min=-5, max=1, step=0.5,
                                       description='Regularization (alpha)', style={'description_width': 'initial'})
epochs_slider = widgets.IntSlider(value=200, min=50, max=1000, step=50,
                                   description='Max epochs', style={'description_width': 'initial'})

run_button = widgets.Button(description='Train Network', button_style='primary')

def on_run_clicked(b):
    with out:
        clear_output(wait=True)
        n1 = hidden_nodes_slider.value
        n2 = second_layer_slider.value
        lr = lr_slider.value
        alpha = alpha_slider.value
        max_iter = epochs_slider.value

        layer_sizes = (n1,) if n2 == 0 else (n1, n2)
        print(f"Training: hidden_layer_sizes={layer_sizes}, lr={lr:.4f}, alpha={alpha:.5f}, max_iter={max_iter}")

        model = MLPClassifier(
            hidden_layer_sizes=layer_sizes,
            activation='logistic',
            alpha=alpha,
            solver='sgd',
            tol=1e-6,
            random_state=1,
            learning_rate_init=lr,
            max_iter=max_iter,
            verbose=False
        )
        model.fit(X_train, Y_train)

        train_acc = accuracy_score(Y_train, model.predict(X_train))
        test_acc = accuracy_score(Y_test, model.predict(X_test))

        print(f"Training accuracy:  {train_acc:.4f}")
        print(f"Test accuracy:      {test_acc:.4f}")
        if train_acc - test_acc > 0.05:
            print("Note: Training accuracy is noticeably higher than test accuracy — this may indicate overfitting.")

        # Plot loss curve
        plt.figure(figsize=(7, 3))
        plt.plot(model.loss_curve_, label='Training loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training Loss Over Epochs')
        plt.legend()
        plt.tight_layout()
        plt.show()

run_button.on_click(on_run_clicked)

display(
    widgets.VBox([
        hidden_nodes_slider,
        second_layer_slider,
        lr_slider,
        alpha_slider,
        epochs_slider,
        run_button,
        out
    ])
)